# 01 — EDA: IEEE-CIS Fraud Detection
**Stack:** pandas · numpy · matplotlib · seaborn

Sections: class imbalance · missing values · amounts · time patterns · categoricals · correlation · missingness signal

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 12,
})

FRAUD_COLOR   = '#E24B4A'
LEGIT_COLOR   = '#378ADD'
NEUTRAL_COLOR = '#888780'

print('Setup complete')

## 1. Load & merge data
> **Before running** — download the dataset:
> ```bash
> pip install kaggle
> kaggle competitions download -c ieee-fraud-detection -p data/raw/
> cd data/raw && unzip ieee-fraud-detection.zip
> ```

In [ ]:
DATA_PATH = '../data/raw/'

print('Loading transactions...')
txn = pd.read_csv(DATA_PATH + 'train_transaction.csv')
print(f'  shape: {txn.shape}')

print('Loading identity...')
ident = pd.read_csv(DATA_PATH + 'train_identity.csv')
print(f'  shape: {ident.shape}')

df = txn.merge(ident, on='TransactionID', how='left')
mem_mb = df.memory_usage(deep=True).sum() / 1e6
print(f'Merged: {df.shape[0]:,} rows x {df.shape[1]} cols  |  {mem_mb:.0f} MB')

## 2. Basic inspection

In [ ]:
print(df.dtypes.value_counts())
df.head(3)

In [ ]:
df.describe().T.sort_values('std', ascending=False).head(20)

## 3. Class imbalance
Fraud datasets are massively imbalanced. **Accuracy is useless** — a model predicting 'not fraud' always scores 99.8%. Use **AUC-PR** as the primary metric.

In [ ]:
fraud_counts = df['isFraud'].value_counts()
fraud_rate   = df['isFraud'].mean() * 100
spw          = int(fraud_counts[0] / fraud_counts[1])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

bars = axes[0].bar(['Legitimate', 'Fraud'], fraud_counts.values,
                   color=[LEGIT_COLOR, FRAUD_COLOR], width=0.5)
for bar, val in zip(bars, fraud_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 1000,
                 f'{val:,}', ha='center', fontsize=11, fontweight='bold')
axes[0].set_title('Transaction count by class', fontweight='bold')
axes[0].set_ylabel('Count')

axes[1].pie(
    fraud_counts.values,
    labels=[f'Legitimate\n{fraud_counts[0]:,}',
            f'Fraud\n{fraud_counts[1]:,}'],
    colors=[LEGIT_COLOR, FRAUD_COLOR],
    autopct='%1.2f%%', startangle=90,
    textprops={'fontsize': 12}
)
axes[1].set_title('Class split', fontweight='bold')

plt.suptitle(f'Fraud rate = {fraud_rate:.2f}%', fontsize=13)
plt.tight_layout()
import os; os.makedirs('../data/external', exist_ok=True)
plt.savefig('../data/external/eda_01_class_imbalance.png',
            dpi=150, bbox_inches='tight')
plt.show()

print(f'Fraud rate         : {fraud_rate:.4f}%')
print(f'Imbalance ratio    : 1:{spw}')
print(f'scale_pos_weight   : {spw}   <- use in LightGBM / XGBoost')
print('Primary metric     : AUC-PR  <- NOT accuracy')

## 4. Missing value analysis

In [ ]:
missing_pct = (df.isnull().mean() * 100).sort_values(ascending=False)
missing_df  = missing_pct[missing_pct > 0].reset_index()
missing_df.columns = ['column', 'missing_pct']

print(f'Cols with any missing : {len(missing_df)}')
print(f'Cols > 50% missing    : {(missing_df.missing_pct > 50).sum()}')
print(f'Cols > 80% missing    : {(missing_df.missing_pct > 80).sum()}')

top30  = missing_df.head(30)
colors = [FRAUD_COLOR if v > 50 else NEUTRAL_COLOR
          for v in top30['missing_pct']]

fig, ax = plt.subplots(figsize=(14, 7))
ax.barh(top30['column'], top30['missing_pct'], color=colors)
ax.axvline(50, color=FRAUD_COLOR, linestyle='--',
           alpha=0.6, label='50% threshold')
ax.axvline(80, color='#A32D2D', linestyle='--',
           alpha=0.6, label='80% threshold')
ax.set_xlabel('Missing %')
ax.set_title('Top 30 columns by % missing', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../data/external/eda_02_missing.png', dpi=150, bbox_inches='tight')
plt.show()

print('Strategy:')
print('  > 80% missing  -> DROP the column')
print('  10-80% missing -> impute + add col_was_missing flag')
print('  < 10% missing  -> impute only')

## 5. Transaction amount distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color, name in [
    (0, LEGIT_COLOR, 'Legitimate'),
    (1, FRAUD_COLOR, 'Fraud')
]:
    s = df[df['isFraud'] == label]['TransactionAmt']
    axes[0].hist(s.clip(upper=s.quantile(0.99)),
                 bins=60, alpha=0.6, color=color,
                 label=name, density=True)
    axes[1].hist(np.log1p(s), bins=60, alpha=0.6,
                 color=color, label=name, density=True)

axes[0].set_title('Amount (clipped 99th pct)', fontweight='bold')
axes[0].set_xlabel('Amount ($)')
axes[0].legend()

axes[1].set_title('log(1 + Amount)', fontweight='bold')
axes[1].set_xlabel('log Amount')
axes[1].legend()

plt.suptitle('Transaction Amount: Fraud vs Legitimate', fontsize=13)
plt.tight_layout()
plt.savefig('../data/external/eda_03_amounts.png', dpi=150, bbox_inches='tight')
plt.show()

print(df.groupby('isFraud')['TransactionAmt'].describe().round(2))
print()
print('Feature idea -> amount_z_score per card (notebook 02)')

## 6. Time-based fraud patterns

In [ ]:
REF_DATE          = pd.Timestamp('2017-12-01')
df['datetime']    = REF_DATE + pd.to_timedelta(df['TransactionDT'], unit='s')
df['hour']        = df['datetime'].dt.hour
df['day_of_week'] = df['datetime'].dt.dayofweek
df['day_name']    = df['datetime'].dt.day_name()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

hourly = df.groupby('hour')['isFraud'].mean() * 100
axes[0].bar(
    hourly.index, hourly.values,
    color=[FRAUD_COLOR if v > hourly.mean()
           else LEGIT_COLOR for v in hourly.values]
)
axes[0].axhline(hourly.mean(), color='black',
                linestyle='--', alpha=0.5, label='Mean')
axes[0].set_title('Fraud rate by hour', fontweight='bold')
axes[0].set_xlabel('Hour (0-23)')
axes[0].set_ylabel('Fraud rate %')
axes[0].legend()

day_order = ['Monday','Tuesday','Wednesday',
             'Thursday','Friday','Saturday','Sunday']
daily = df.groupby('day_name')['isFraud'].mean()\
          .reindex(day_order) * 100
axes[1].bar(
    daily.index, daily.values,
    color=[FRAUD_COLOR if v > daily.mean()
           else LEGIT_COLOR for v in daily.values]
)
axes[1].axhline(daily.mean(), color='black', linestyle='--', alpha=0.5)
axes[1].set_title('Fraud rate by day of week', fontweight='bold')
axes[1].set_ylabel('Fraud rate %')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('Time-based Fraud Patterns', fontsize=13)
plt.tight_layout()
plt.savefig('../data/external/eda_04_time.png', dpi=150, bbox_inches='tight')
plt.show()

night_hours = list(range(0, 6)) + [23]
night_mask  = df['hour'].isin(night_hours)
print(f'Night (11pm-6am) fraud rate : '
      f"{df[night_mask]['isFraud'].mean()*100:.2f}%")
print(f'Day   (6am-11pm) fraud rate : '
      f"{df[~night_mask]['isFraud'].mean()*100:.2f}%")
print()
print('Features to add -> is_night, is_weekend, hour, day_of_week')

## 7. Fraud rate by categorical features

In [ ]:
all_cats = ['ProductCD','card4','card6',
            'P_emaildomain','R_emaildomain']
cat_cols = [c for c in all_cats if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
flat  = axes.flatten()
avg   = df['isFraud'].mean() * 100

for ax, col in zip(flat, cat_cols):
    stats = (
        df.groupby(col)['isFraud']
          .agg(['mean', 'count'])
          .rename(columns={'mean':'fraud_rate','count':'n'})
          .sort_values('fraud_rate', ascending=False)
          .head(12)
    )
    stats['fraud_rate'] *= 100
    ax.bar(
        stats.index.astype(str),
        stats['fraud_rate'],
        color=[FRAUD_COLOR if v > avg else LEGIT_COLOR
               for v in stats['fraud_rate']]
    )
    ax.axhline(avg, color='black', linestyle='--',
               alpha=0.4, label='Overall avg')
    ax.set_title(col, fontweight='bold')
    ax.set_ylabel('Fraud rate %')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(fontsize=8)

for ax in flat[len(cat_cols):]:
    ax.set_visible(False)

plt.suptitle('Fraud rate by categorical feature', fontsize=13)
plt.tight_layout()
plt.savefig('../data/external/eda_05_categoricals.png',
            dpi=150, bbox_inches='tight')
plt.show()

## 8. Correlation heatmap

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns
usable   = [
    col for col in num_cols
    if df[col].isnull().mean() < 0.4
    and df[col].std() > 0
    and col not in ('TransactionID', 'TransactionDT')
]

corr_target = (
    df[usable].corr()['isFraud']
    .drop('isFraud').abs()
    .sort_values(ascending=False)
)
top20 = corr_target.head(20).index.tolist()
matrix = df[top20 + ['isFraud']].corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(matrix, dtype=bool))
sns.heatmap(
    matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.4, ax=ax, cbar_kws={'shrink': 0.8}
)
ax.set_title('Top 20 features correlated with isFraud',
             fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('../data/external/eda_06_correlation.png',
            dpi=150, bbox_inches='tight')
plt.show()

print('Top 10 by |corr| with isFraud:')
print(corr_target.head(10).to_string())

## 9. Missingness as a fraud signal
> In IEEE-CIS, **missing device info is itself a fraud indicator**.
> Fraudsters avoid using identifiable devices.

In [ ]:
mid_cols = [
    col for col in df.columns
    if 0.10 < df[col].isnull().mean() < 0.80
][:12]

rows = []
for col in mid_cols:
    mfr  = df[df[col].isnull()]['isFraud'].mean() * 100
    pfr  = df[df[col].notna()]['isFraud'].mean() * 100
    lift = mfr / (pfr + 1e-9)
    rows.append({
        'feature'          : col,
        'fraud_if_missing' : round(mfr, 2),
        'fraud_if_present' : round(pfr, 2),
        'lift'             : round(lift, 2),
    })

flag_df = pd.DataFrame(rows).sort_values('lift', ascending=False)

fig, ax = plt.subplots(figsize=(13, 5))
x, w = np.arange(len(flag_df)), 0.38
ax.bar(x - w/2, flag_df['fraud_if_missing'],
       w, label='When missing', color=FRAUD_COLOR)
ax.bar(x + w/2, flag_df['fraud_if_present'],
       w, label='When present', color=LEGIT_COLOR)
ax.set_xticks(x)
ax.set_xticklabels(flag_df['feature'], rotation=35, ha='right')
ax.set_ylabel('Fraud rate %')
ax.set_title('Missingness as a fraud signal', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../data/external/eda_07_missing_signal.png',
            dpi=150, bbox_inches='tight')
plt.show()

print(flag_df.to_string(index=False))
print()
print('For lift > 1.5 -> create col_was_missing binary flag feature')

## 10. Key findings & next steps

In [ ]:
fr    = df['isFraud'].mean() * 100
ratio = int(df['isFraud'].value_counts()[0]
            / df['isFraud'].value_counts()[1])
fraud_amt = df[df['isFraud']==1]['TransactionAmt'].mean()
legit_amt = df[df['isFraud']==0]['TransactionAmt'].mean()
hi_miss   = (df.isnull().mean() > 0.80).sum()
mid_miss  = ((df.isnull().mean() > 0.10)
             & (df.isnull().mean() < 0.80)).sum()

print('=' * 58)
print('  EDA KEY FINDINGS')
print('=' * 58)
print()
print('1. CLASS IMBALANCE')
print(f'   Fraud rate  : {fr:.3f}%   Ratio 1:{ratio}')
print(f'   scale_pos_weight = {ratio}  <-- set in LightGBM')
print('   Primary metric   = AUC-PR  <-- not accuracy')
print()
print('2. MISSING VALUES')
print(f'   > 80% missing : {hi_miss} cols  -> DROP')
print(f'   10-80% miss   : {mid_miss} cols  -> FLAG + IMPUTE')
print()
print('3. TIME PATTERNS')
print('   Night (11pm-6am) -> elevated fraud')
print('   Add: is_night, is_weekend, hour, day_of_week')
print()
print('4. AMOUNT PATTERNS')
print(f'   Fraud mean : {fraud_amt:.2f}   Legit mean : {legit_amt:.2f}')
print('   Add: amount_z_score per card, log_amount')
print()
print('5. CATEGORICAL SIGNALS')
print('   ProductCD, card4, card6 -> strong fraud rate variation')
print('   Use target encoding in notebook 02')
print()
print('NEXT: open 02_feature_engineering.ipynb')
print('=' * 58)